This notebook answers the research question: which embeddings predict a field's crop better, AEF, OlmoEarth, or both together? It trains one LightGBM classifier per feature set on training fields from every country, changing only the features, and scores each one on the held-out test blocks of every country.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import pyarrow as pa
import lightgbm as lgb
import ultraplot as uplt
from pathlib import Path
from IPython.display import display
from sklearn.metrics import f1_score
pd.set_option('display.max_rows',20)
warnings.filterwarnings('ignore',message="The argument 'eval_set' is deprecated")

In [ ]:
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
FIGDIR      = Path('../figs')
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
FEATURESETS = {'aef':[f'aef{i:03d}' for i in range(128)],
               'olmoearth':[f'olmoearth{i:03d}' for i in range(128)],
               'both':[f'aef{i:03d}' for i in range(128)]+[f'olmoearth{i:03d}' for i in range(128)]}
COLORS      = {'aef':'#2a78d6','olmoearth':'#eb6834','both':'#1baf7a'}
TRAINFIELDS = 20000
VALFIELDS   = 5000
EARLYSTOP   = 50
TOPCROPS    = 25
SEED        = 42
NCORES      = len(os.sched_getaffinity(0))
PARAMS      = dict(n_estimators=2000,learning_rate=0.1,num_leaves=31,max_bin=63,colsample_bytree=0.5,class_weight='balanced',
                   metric='multi_error',random_state=SEED,n_jobs=NCORES,verbose=-1)

pa.set_cpu_count(NCORES)
pa.set_io_thread_count(NCORES)

`TRAINFIELDS` and `VALFIELDS` are the fields sampled from each country's training and validation blocks, so every country contributes equally to the model. Every field in the test blocks is scored. `PARAMS` are the LightGBM settings: `class_weight='balanced'` makes rare crops count as much as common ones, and training stops once the error rate on the validation fields has not improved for `EARLYSTOP` rounds. `TOPCROPS` is how many crops the confusion-matrix figure shows.

In [ ]:
def load_collection(collection):
    table  = pd.read_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet")
    blocks = table.chip.str[4:-4]+'_'+(table.chip.str[-4:-2].astype(int)//6).astype(str)+'_'+(table.chip.str[-2:].astype(int)//6).astype(str)
    mixed  = int((table.groupby(blocks).split.nunique()>1).sum())
    sample = pd.concat([table[table.split=='train'].sample(min(TRAINFIELDS,(table.split=='train').sum()),random_state=SEED),
                        table[table.split=='val'].sample(min(VALFIELDS,(table.split=='val').sum()),random_state=SEED),
                        table[table.split=='test']])
    return sample.assign(country=collection.split('/')[1]),{'blocks':blocks.nunique(),'mixedblocks':mixed}

In [ ]:
loaded = [load_collection(collection) for collection in COLLECTIONS]
fields = pd.concat([sample for sample,_ in loaded],ignore_index=True)
blocks = pd.DataFrame([check for _,check in loaded])

print(f'Loaded {len(fields):,} fields from {fields.country.nunique()} countries...')
print(f"- {(fields.split=='train').sum():,} training, {(fields.split=='val').sum():,} validation and {(fields.split=='test').sum():,} test fields, {fields.crop.nunique()} crops")
print(f'- {blocks.mixedblocks.sum()} of {blocks.blocks.sum():,} 6 km blocks have fields in more than one split (should be 0)')

In [ ]:
train = fields[fields.split=='train']
val   = fields[(fields.split=='val')&fields.crop.isin(train.crop.unique())]
test  = fields.loc[fields.split=='test',['id','country','crop']].copy()
print(f'Training one model per feature set on {len(train):,} fields ({train.crop.nunique()} crops)...')

for featureset,cols in FEATURESETS.items():
    start = time.perf_counter()
    model = lgb.LGBMClassifier(**PARAMS).fit(train[cols],train.crop,eval_set=[(val[cols],val.crop)],callbacks=[lgb.early_stopping(EARLYSTOP,verbose=False)])
    test[featureset] = model.predict(fields.loc[fields.split=='test',cols])
    minutes,seconds  = divmod(round(time.perf_counter()-start),60)
    print(f"- {featureset}: macro F1 {f1_score(test.crop,test[featureset],average='macro'):.3f}, accuracy {(test.crop==test[featureset]).mean():.3f}, {model.best_iteration_} rounds ({minutes}m {seconds}s)")

test.to_parquet(PROCDIR/'results_predictions.parquet',index=False)

## Results

In [ ]:
metrics = pd.DataFrame([{'country':country,'featureset':featureset,'accuracy':(group.crop==group[featureset]).mean(),
                         'macrof1':f1_score(group.crop,group[featureset],labels=group.crop.unique(),average='macro',zero_division=0)}
                        for country,group in [('all',test)]+list(test.groupby('country')) for featureset in FEATURESETS])
metrics.to_csv(PROCDIR/'results_metrics.csv',index=False)
table   = metrics.pivot(index='country',columns='featureset',values='macrof1')[list(FEATURESETS)]
wins    = table.drop(index='all').idxmax(axis=1).value_counts().reindex(list(FEATURESETS),fill_value=0)

print('Macro F1 on each country\'s test fields ("all" pools every country)...')
display(table.style.format('{:.3f}').highlight_max(axis=1,props='font-weight:bold'))
print(f"Highest macro F1 by country: {', '.join(f'{featureset} {count}' for featureset,count in wins.items())}")

In [ ]:
percountry = table.drop(index='all').sort_values('both')

fig,ax = uplt.subplots(refwidth=4,refheight=0.25*len(percountry))
for featureset,color in COLORS.items():
    ax.scatter(percountry[featureset],percountry.index,color=color,label=featureset,s=25)
ax.format(xlabel='Macro F1',ylabel='',title='Macro F1 on held-out test blocks',ygrid=True,xlim=(0,1))
ax.legend(loc='b',ncols=3,frame=False)
fig.savefig(FIGDIR/'03_macrof1_by_country.png',dpi=200)

In [ ]:
crops    = sorted(test.crop.unique())
perclass = pd.DataFrame({featureset:f1_score(test.crop,test[featureset],labels=crops,average=None,zero_division=0) for featureset in FEATURESETS},index=crops)
perclass = perclass.assign(testfields=test.crop.value_counts()).sort_values('testfields',ascending=False)
perclass.rename_axis('crop').to_csv(PROCDIR/'results_perclass.csv')
pairs    = pd.concat([test[test.crop!=test[featureset]].value_counts(['crop',featureset]).head(10).rename('fields').reset_index()
                      .rename(columns={'crop':'true',featureset:'predicted'}).assign(featureset=featureset) for featureset in FEATURESETS])
pairs.to_csv(PROCDIR/'results_topconfusions.csv',index=False)

print(f"F1 per crop for the {min(10,len(perclass))} most common crops (all {len(perclass)} are in {PROCDIR/'results_perclass.csv'})...")
display(perclass.head(10).style.format({featureset:'{:.3f}' for featureset in FEATURESETS}|{'testfields':'{:,}'}))
print('Most confused crop pairs with both feature sets...')
display(pairs[pairs.featureset=='both'].drop(columns='featureset').style.format({'fields':'{:,}'}).hide(axis='index'))

In [ ]:
top = test.crop.value_counts().head(TOPCROPS).index

fig,axs = uplt.subplots(ncols=3,refwidth=4,share=False)
for ax,featureset in zip(axs,FEATURESETS):
    counts = pd.crosstab(test.crop,test[featureset])
    matrix = counts.reindex(index=top,columns=top,fill_value=0).div(counts.loc[top].sum(axis=1),axis=0)
    ax.imshow(matrix.to_numpy(),cmap='Blues',vmin=0,vmax=1)
    ax.format(title=featureset,xlabel='Predicted crop',ylabel='True crop',grid=False,xticks=range(len(top)),yticks=range(len(top)),
              xticklabels=list(top),yticklabels=list(top) if featureset=='aef' else [],xrotation=90,ticklabelsize=6,xtickminor=False,ytickminor=False)
fig.format(suptitle=f'Share of each true crop predicted as each crop ({TOPCROPS} most common crops, all countries)')
fig.savefig(FIGDIR/'03_confusion.png',dpi=200)

## Data issues

Already handled upstream: notebook 01 drops the four collections with coarse labels (Croatia, Lithuania, Valencia and Castilla-La Mancha), and notebook 02 drops catch-all classes (`not_known_and_other`, `other_arable_land_crops` and `unspecified_*`). Remaining caveats, with counts in `data/interim/build_log.csv`:

1. **Many tiny fields in some countries.** Fields under 5 pixels are dropped, e.g. 473 k in the Netherlands and 372 k in Austria, so small fields are under-represented.
2. **Duplicate field ids.** Czechia (7,389) and Catalonia (14,335) have fields sharing an id. They are dropped because their labels are ambiguous.
3. **Fields outside the chips.** Fields whose centroid falls outside every 2 km chip have no split and are dropped.
4. **OlmoEarth uses half its dimensions.** FTW stores only the first 64 of OlmoEarth's 128 dimensions. AEF also has 64, so the comparison is like-for-like, but OlmoEarth may be handicapped.
5. **Country-specific crops.** Some crops exist in only one country, so the pooled model learns them from that country alone.